# Laboratório 10 — Pipeline Definitivo (RAG + QLoRA + Otimização de Inferência)

**Aluno:** alcivan  
**Objetivo:** Salvar o Transformer do colapso de VRAM combinando **QLoRA (4-bit)**, **KV Cache** e **FlashAttention-2 / SDPA**.

> ⚠️ **Aviso sobre a GPU:** FlashAttention-2 só roda em GPUs Ampere ou mais novas (A100, L4, RTX 30/40). A **Tesla T4** do Colab grátis é Turing (SM 7.5) e **não** suporta `flash_attention_2`. Este notebook tenta carregar `flash_attention_2`; se falhar, faz *fallback* automático para `sdpa` (PyTorch Scaled-Dot-Product-Attention, com kernel memory-efficient), que demonstra o mesmo conceito hardware-aware.

## Passo 0 — Instalação de dependências (Colab)
Se rodando localmente e o ambiente já tem essas libs, pode pular.

In [ ]:
!pip -q install -U "transformers>=4.44" "accelerate>=0.33" "bitsandbytes>=0.43" "sentencepiece" "protobuf"
# flash-attn é opcional; só instala se tiver GPU Ampere+. Em T4 isto pode até instalar mas falhará em runtime.
# !pip -q install flash-attn --no-build-isolation

In [ ]:
import torch, time, gc, os
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

assert torch.cuda.is_available(), "CUDA não disponível. Use uma GPU (Colab → Runtime → Change runtime type → T4)."
device = torch.device("cuda")
gpu_name = torch.cuda.get_device_name(0)
cc_major, cc_minor = torch.cuda.get_device_capability(0)
print(f"GPU: {gpu_name} | Compute Capability: {cc_major}.{cc_minor}")
SUPPORTS_FA2 = cc_major >= 8
print(f"Suporta FlashAttention-2 nativo? {SUPPORTS_FA2}")

## Passo 1 — Ingestão Eficiente (QLoRA 4-bit)
Carrega `TinyLlama-1.1B-Chat-v1.0` quantizado em 4-bit via `bitsandbytes`.  
**Métrica:** VRAM ocupada após o load.

In [ ]:
MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

def free_vram():
    gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()

def mb(x):
    return x / (1024**2)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

free_vram()
model_baseline = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map={"": 0},
    attn_implementation="eager",  # baseline sem otimização hardware-aware
)
model_baseline.eval()

vram_load_mb = mb(torch.cuda.memory_allocated())
print(f"[Passo 1] VRAM ocupada pelo modelo 4-bit: {vram_load_mb:,.1f} MB")

## Passo 2 — Simulando o RAG Massivo
Gera um contexto fictício de ~12.000 tokens simulando 5 capítulos médicos recuperados pelo banco vetorial.

In [ ]:
# NOTA SOBRE TAMANHO DO CONTEXTO:
# O enunciado fala em 10-15k tokens, mas em T4 (16GB) com eager-attention SEM kv-cache
# isso estoura a VRAM (a matriz n×n de attention em fp16 com n=12000 pede ~9GB só para
# ela). Para COMPARAR o baseline com o otimizado precisamos que o baseline complete.
# Solução: usar TARGET_TOKENS=4000 — ainda dispara o O(n²) de forma evidente nas métricas,
# mas cabe em T4. Em GPU >=24GB (A100/L4/RTX3090), pode subir para 12000 e ver o efeito real.
TARGET_TOKENS = 4000

fake_chapter = (
    "Capítulo de Manual Médico. Paciente do sexo masculino, 58 anos, hipertenso, "
    "apresenta dispneia aos médios esforços há três semanas, com episódios de dor "
    "precordial em aperto, irradiando para o membro superior esquerdo. Ausculta "
    "cardíaca revela sopro sistólico em foco aórtico. Eletrocardiograma evidencia "
    "alterações de repolarização ventricular em parede inferior. Ecocardiograma "
    "transtorácico demonstra hipertrofia concêntrica do ventrículo esquerdo, com "
    "fração de ejeção preservada em 58%. Cateterismo coronariano mostra lesão "
    "obstrutiva de 80% em artéria descendente anterior proximal. Conduta: "
    "angioplastia transluminal coronariana com implante de stent farmacológico, "
    "associada a dupla antiagregação plaquetária por 12 meses, estatina de alta "
    "intensidade, betabloqueador e inibidor da ECA. Acompanhamento ambulatorial "
    "trimestral com avaliação de função renal, perfil lipídico e ecocardiograma anual. "
)

rag_text = fake_chapter
while len(tokenizer(rag_text, return_tensors="pt").input_ids[0]) < TARGET_TOKENS:
    rag_text += fake_chapter

prompt = (
    "Você é um assistente clínico. Com base nos capítulos abaixo, gere um resumo "
    "clínico estruturado.\n\n--- CONTEXTO RAG ---\n" + rag_text + "\n--- FIM ---\n\nResumo clínico:\n"
)
inputs = tokenizer(prompt, return_tensors="pt").to(device)
n_ctx = inputs.input_ids.shape[1]
print(f"[Passo 2] Tokens no contexto (prompt + RAG): {n_ctx:,}")

## Passo 3 — Gargalo de Geração (SEM KV Cache)
Força `use_cache = False`. Cada novo token re-projeta Q, K, V de toda a sequência.  
**Métricas:** tempo total + pico de VRAM.

In [ ]:
NEW_TOKENS = 100

def benchmark_generate(model, inputs, use_cache, label):
    """Roda generate e captura métricas. Se der OOM, registra como tal e segue."""
    model.config.use_cache = use_cache
    free_vram()
    torch.cuda.synchronize()
    t0 = time.time()
    try:
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=NEW_TOKENS,
                do_sample=False,
                use_cache=use_cache,
                pad_token_id=tokenizer.eos_token_id,
            )
        torch.cuda.synchronize()
        dt = time.time() - t0
        peak_mb = mb(torch.cuda.max_memory_allocated())
        n_gen = out.shape[1] - inputs.input_ids.shape[1]
        print(f"[{label}] tempo={dt:.2f}s | tokens/s={n_gen/dt:.2f} | pico VRAM={peak_mb:,.1f} MB")
        return {"label": label, "time_s": dt, "tok_per_s": n_gen/dt, "peak_vram_mb": peak_mb, "oom": False}
    except torch.cuda.OutOfMemoryError as e:
        peak_mb = mb(torch.cuda.max_memory_allocated())
        free_vram()
        print(f"[{label}] *** OOM *** pico VRAM antes do crash={peak_mb:,.1f} MB\n  -> {str(e).splitlines()[0]}")
        return {"label": label, "time_s": float('nan'), "tok_per_s": 0.0, "peak_vram_mb": peak_mb, "oom": True}

results = []
results.append(benchmark_generate(model_baseline, inputs, use_cache=False, label="Baseline (eager, NO KV cache)"))

## Passo 4 — Engenharia de Otimização
1. Liga **KV Cache** (`use_cache=True`).  
2. Recarrega o modelo com **FlashAttention-2** se a GPU suportar; senão **SDPA**.

In [ ]:
# 4.1 — Mesma config (eager) mas com KV cache ligado, para isolar o ganho do cache
results.append(benchmark_generate(model_baseline, inputs, use_cache=True, label="Eager + KV Cache"))

# Liberar o baseline antes de carregar o otimizado
del model_baseline
free_vram()

In [ ]:
# 4.2 — Recarrega com attention hardware-aware
chosen_attn = "flash_attention_2" if SUPPORTS_FA2 else "sdpa"
print(f"Tentando attn_implementation='{chosen_attn}'...")
try:
    model_opt = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map={"": 0},
        attn_implementation=chosen_attn,
    )
except Exception as e:
    print(f"Falha com {chosen_attn}: {e}\nFazendo fallback para 'sdpa'.")
    chosen_attn = "sdpa"
    model_opt = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map={"": 0},
        attn_implementation="sdpa",
    )
model_opt.eval()
print(f"[Passo 4] Modelo recarregado com attn_implementation='{chosen_attn}'")

results.append(benchmark_generate(
    model_opt, inputs, use_cache=True,
    label=f"{chosen_attn.upper()} + KV Cache (OTIMIZADO)"
))

## Resumo dos benchmarks

In [ ]:
import math
print(f"\nVRAM do modelo 4-bit carregado: {vram_load_mb:,.1f} MB")
print(f"Tokens de contexto (RAG simulado): {n_ctx:,}")
print(f"Attention otimizada usada: {chosen_attn}\n")
print(f"{'Configuração':<45} {'Tempo(s)':>10} {'tok/s':>8} {'Pico VRAM(MB)':>16}")
print("-" * 82)
for r in results:
    t = "OOM" if r['oom'] else f"{r['time_s']:.2f}"
    ts = "—"   if r['oom'] else f"{r['tok_per_s']:.2f}"
    print(f"{r['label']:<45} {t:>10} {ts:>8} {r['peak_vram_mb']:>16,.1f}")

base = results[0]
opt  = results[-1]
if not base['oom'] and not opt['oom']:
    print(f"\nSpeedup tempo (baseline → otimizado): {base['time_s']/opt['time_s']:.2f}x")
if base['peak_vram_mb'] > 0:
    print(f"Redução pico VRAM:                    {(1 - opt['peak_vram_mb']/base['peak_vram_mb'])*100:.1f}%")

In [ ]:
# (Opcional) gráfico de barras das métricas
try:
    import matplotlib.pyplot as plt
    labels = [r['label'] for r in results]
    times  = [r['time_s'] for r in results]
    vrams  = [r['peak_vram_mb'] for r in results]
    fig, axes = plt.subplots(1, 2, figsize=(13,4))
    axes[0].bar(range(len(labels)), times); axes[0].set_title('Tempo (s)')
    axes[0].set_xticks(range(len(labels))); axes[0].set_xticklabels(labels, rotation=20, ha='right')
    axes[1].bar(range(len(labels)), vrams); axes[1].set_title('Pico VRAM (MB)')
    axes[1].set_xticks(range(len(labels))); axes[1].set_xticklabels(labels, rotation=20, ha='right')
    plt.tight_layout(); plt.savefig('benchmark.png', dpi=120); plt.show()
except Exception as e:
    print('matplotlib opcional:', e)